In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

WindowsPath('c:/Users/Dell - i5 11th Gen/Desktop/atm-protein-conservation-explorer')

In [3]:
variant_scores_file = project_root / "data" / "processed" / "variant_scores.csv"
position_scores_file = project_root / "data" / "processed" / "position_scores.csv"

variant_scores = pd.read_csv(variant_scores_file)
position_scores = pd.read_csv(position_scores_file)

print(f"Position rows: {len(position_scores)}")
print(f"Variant rows: {len(variant_scores)}")

variant_scores.head()


Position rows: 1793
Variant rows: 4526


,Unnamed: 0,gene,clinvar_variation_id,vcv_accession,rs_id,clinvar_variation,transcript,hgvs_c,hgvs_p,protein_change,...,usable_species_count,matching_species_count,gap_species_count,ambiguous_species_count,conservation_score,conservation_percent,reference_matches_human,alternate_species_count,alternate_species_fraction,alternate_seen_in_species
0,0,ATM,4170131,VCV004170131,rs4279232,NM_000051.4(ATM):c.4A>C (p.Ser2Arg),NM_000051.4,c.4A>C,p.Ser2Arg,S2R,...,814,761,12,0,0.934889,93.488943,True,0,0.000000,False
1,1,ATM,640846,VCV000640846,rs639367,NM_000051.4(ATM):c.4A>G (p.Ser2Gly),NM_000051.4,c.4A>G,p.Ser2Gly,S2G,...,814,761,12,0,0.934889,93.488943,True,0,0.000000,False
2,2,ATM,181943,VCV000181943,rs180377,NM_000051.4(ATM):c.5G>C (p.Ser2Thr),NM_000051.4,c.5G>C,p.Ser2Thr,S2T,...,814,761,12,0,0.934889,93.488943,True,6,0.007371,True
3,3,ATM,922075,VCV000922075,rs911284,NM_000051.4(ATM):c.6T>A (p.Ser2Arg),NM_000051.4,c.6T>A,p.Ser2Arg,S2R,...,814,761,12,0,0.934889,93.488943,True,0,0.000000,False
4,4,ATM,1337452,VCV001337452,rs1328461,NM_000051.4(ATM):c.6T>G (p.Ser2Arg),NM_000051.4,c.6T>G,p.Ser2Arg,S2R,...,814,761,12,0,0.934889,93.488943,True,0,0.000000,False


In [5]:
required_columns = {
    "protein_position", "human_residue", "conservation_score", "conservation_percent", "usable_species_count"
}
missing_columns = required_columns - set(position_scores.columns)

if missing_columns:
    raise ValueError(f"Missing position columns: {sorted(missing_columns)}")

if position_scores["protein_position"].duplicated().any():
    raise ValueError(f"Duplication protein position found in position_scores")

if not position_scores["conservation_score"].between(0, 1).all():
    raise ValueError("Conservation scores outside 0-1.")

print("Position scores OK.")

Position scores OK.


In [ ]:
# left out the N-terminal HEAT-repeat region (~residues 1-1939) for now
atm_domains = {
    "FAT": (1940, 2566),
    "PI3K/PI4K catalytic": (2686, 2998),
    "FATC": (3024, 3056),
}

domain_colors = {
    "FAT": "#fbd38d",
    "PI3K/PI4K catalytic": "#feb2b2",
    "FATC": "#9ae6b4",
}

def assign_domain(position):
    for name, (start, end) in atm_domains.items():
        if start <= position <= end:
            return name
    return "Other"